# Exercise 3.1: Mainz OSM Tool-Using LLM Widget

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yfeng-hsm/KI_Geodatenanalyse_SS26/blob/main/lectures/03_llm_basics/notebooks/exercise_3_1_llm_osm_tool_widget_mainz.ipynb)

This notebook builds a small conversational geospatial widget for Mainz. A Uni Mainz KI-Chat@JGU model translates natural-language questions into a restricted JSON tool plan. Python validates the plan, calls Overpass or OpenRouteService, runs small spatial analyses, and visualizes the result on a Folium map.

The model proposes a plan. Python validates and executes it. The model never runs arbitrary code.

## Learning Outcomes

After this exercise you should be able to:

- Call the OpenAI-compatible Uni Mainz KI-Chat@JGU API.
- Ask an LLM to generate structured JSON for a geospatial task.
- Validate LLM output before calling external APIs.
- Query OpenStreetMap features in Mainz with Overpass QL.
- Use GeoPandas and Shapely for buffer and nearest-feature analysis.
- Optionally call OpenRouteService for route analysis.
- Visualize and test the results in Colab.

## 1. Colab Setup

Run this cell first. The notebook is designed for Google Colab.

In [ ]:
!pip -q install geopandas shapely pyproj folium ipywidgets requests pandas


## 2. Imports and Study Area

The notebook uses `EPSG:4326` for APIs and maps, and `EPSG:25832` for metric distance calculations around Mainz.

In [ ]:
from __future__ import annotations

import getpass
import json
import re
import time
from typing import Any

import geopandas as gpd
import pandas as pd
import requests
from IPython.display import Markdown, display
from shapely.geometry import Point, shape

import folium
from folium.plugins import MarkerCluster

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

WGS84 = "EPSG:4326"
METRIC_CRS = "EPSG:25832"

MAINZ_BBOX = {"south": 49.90, "west": 8.13, "north": 50.05, "east": 8.36}
MAINZ_CENTER = (49.9929, 8.2473)  # lat, lon

KNOWN_PLACES = {
    "mainz_hbf": {"label": "Mainz Hauptbahnhof", "lat": 50.0010, "lon": 8.2587},
    "jgu": {"label": "Johannes Gutenberg University Mainz", "lat": 49.9936, "lon": 8.2419},
    "mainz_dom": {"label": "Mainz Cathedral", "lat": 49.9995, "lon": 8.2742},
    "mainz_theater": {"label": "Staatstheater Mainz", "lat": 50.0002, "lon": 8.2714},
}

ALLOWED_TOOLS = {"overpass_search", "osm_buffer_search", "osm_nearest", "ors_route"}
ALLOWED_OSM_KEYS = {"amenity", "tourism", "shop", "leisure", "public_transport", "railway", "highway"}

OVERPASS_URL = "https://overpass-api.de/api/interpreter"
KI_CHAT_API_BASE_URL = "https://ki-chat.uni-mainz.de/api"

print("Mainz study area loaded.")
print("Known places:", ", ".join(KNOWN_PLACES))


## 3. Uni Mainz KI-Chat API

Create an API key in the KI-Chat@JGU web interface and paste it below. Do not store API keys in notebook cells.

Official documentation: https://www.zdv.uni-mainz.de/ki-chat-api-nutzung/

In [ ]:
KI_CHAT_API_KEY = getpass.getpass("Paste your KI-Chat@JGU API key for this session: ").strip()
if not KI_CHAT_API_KEY:
    raise RuntimeError("No API key entered.")

ki_chat_headers = {
    "Authorization": f"Bearer {KI_CHAT_API_KEY}",
    "Content-Type": "application/json",
}
print("KI-Chat API key loaded for this notebook session.")


In [ ]:
def ki_chat_request(method: str, endpoint: str, **kwargs: Any) -> Any:
    url = f"{KI_CHAT_API_BASE_URL}{endpoint}"
    response = requests.request(method, url, headers=ki_chat_headers, timeout=90, **kwargs)
    if not response.ok:
        print(f"HTTP {response.status_code} for {method} {endpoint}")
        try:
            print(json.dumps(response.json(), indent=2, ensure_ascii=False))
        except ValueError:
            print(response.text[:1000])
        response.raise_for_status()
    return response.json()


def preview_json(data: Any, max_chars: int = 3000) -> None:
    text = json.dumps(data, indent=2, ensure_ascii=False)
    print(text[:max_chars] + ("\n..." if len(text) > max_chars else ""))


In [ ]:
models_response = ki_chat_request("GET", "/models")
model_ids = [m.get("id") for m in models_response.get("data", []) if m.get("id")]
print("Available model IDs:")
for model_id in model_ids:
    print("-", model_id)

DEFAULT_CHAT_MODEL = "GPT OSS 120B"
chat_model = DEFAULT_CHAT_MODEL if DEFAULT_CHAT_MODEL in model_ids else (model_ids[0] if model_ids else DEFAULT_CHAT_MODEL)
print(f"\nUsing chat model: {chat_model}")


## 4. Tool Plan Contract

The LLM must return exactly one JSON object. Python will reject unsafe or incomplete plans.

Supported tools:

- `overpass_search`: query OSM features inside the Mainz bounding box.
- `osm_buffer_search`: query OSM features and filter them to a radius around a Mainz point.
- `osm_nearest`: query OSM features and return the nearest features to a Mainz point.
- `ors_route`: request a route between two known Mainz places.

Example JSON plan:

```json
{
  "tool": "osm_buffer_search",
  "question": "Find cafes within 800 meters of Mainz Hauptbahnhof.",
  "osm_tags": {"amenity": "cafe"},
  "center": {"place_id": "mainz_hbf", "label": "Mainz Hauptbahnhof", "lat": 50.001, "lon": 8.2587},
  "radius_m": 800,
  "overpass_ql": "[out:json][timeout:25]; ... out center tags;",
  "analysis": "buffer_count",
  "map_title": "Cafes near Mainz Hauptbahnhof"
}
```

In [ ]:
TOOL_PLANNER_SYSTEM_PROMPT = f"""
You are a geospatial planning assistant for a teaching notebook.
Return exactly one JSON object and no Markdown.

Scope:
- The study area is Mainz, Germany only.
- Use this Mainz bbox in Overpass queries: south={MAINZ_BBOX['south']}, west={MAINZ_BBOX['west']}, north={MAINZ_BBOX['north']}, east={MAINZ_BBOX['east']}.
- Never create Germany-wide, Europe-wide, or global queries.
- Maximum radius_m is 5000.
- Prefer known place IDs: {', '.join(KNOWN_PLACES.keys())}.

Allowed tools: overpass_search, osm_buffer_search, osm_nearest, ors_route.
Allowed OSM keys: amenity, tourism, shop, leisure, public_transport, railway, highway.

For OSM tools include: tool, question, osm_tags, overpass_ql, analysis, map_title.
For radius or nearest tasks also include: center and radius_m or limit.
For ors_route include: tool, question, start_place_id, end_place_id, profile, map_title.
Route profiles: foot-walking, cycling-regular, driving-car.

Overpass QL requirements:
- Include [out:json][timeout:25];
- Use node/way/relation clauses with the Mainz bbox or an around radius.
- End with out center tags;
""".strip()


def extract_json_object(text: str) -> dict[str, Any]:
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?", "", text).strip()
        text = re.sub(r"```$", "", text).strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))


def generate_tool_plan(question: str, model: str | None = None) -> dict[str, Any]:
    payload = {
        "model": model or chat_model,
        "messages": [
            {"role": "system", "content": TOOL_PLANNER_SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        "temperature": 0.1,
        "max_tokens": 1000,
    }
    response = ki_chat_request("POST", "/chat/completions", json=payload)
    content = response["choices"][0]["message"]["content"]
    plan = extract_json_object(content)
    validate_tool_plan(plan)
    return plan


## 5. Validate LLM Output

Treat LLM output as untrusted. Validation prevents over-broad queries and unsupported tool calls.

In [ ]:
class PlanValidationError(ValueError):
    pass


def point_is_inside_mainz(lat: float, lon: float, margin: float = 0.03) -> bool:
    return (
        MAINZ_BBOX["south"] - margin <= lat <= MAINZ_BBOX["north"] + margin
        and MAINZ_BBOX["west"] - margin <= lon <= MAINZ_BBOX["east"] + margin
    )


def validate_osm_tags(osm_tags: dict[str, Any]) -> None:
    if not isinstance(osm_tags, dict) or not osm_tags:
        raise PlanValidationError("osm_tags must be a non-empty dictionary.")
    for key, value in osm_tags.items():
        if key not in ALLOWED_OSM_KEYS:
            raise PlanValidationError(f"OSM key {key!r} is not allowed.")
        if value is not True and not isinstance(value, str):
            raise PlanValidationError(f"OSM value for {key!r} must be a string or true.")


def validate_center(center: dict[str, Any]) -> None:
    if not isinstance(center, dict):
        raise PlanValidationError("center must be a dictionary.")
    lat = float(center.get("lat"))
    lon = float(center.get("lon"))
    if not point_is_inside_mainz(lat, lon):
        raise PlanValidationError(f"Center point is outside Mainz: {(lat, lon)}")


def validate_overpass_ql(overpass_ql: str) -> None:
    if not isinstance(overpass_ql, str) or not overpass_ql.strip():
        raise PlanValidationError("overpass_ql must be a non-empty string.")
    ql = overpass_ql.lower()
    if "[out:json" not in ql:
        raise PlanValidationError("Overpass QL must request JSON output.")
    if "out" not in ql:
        raise PlanValidationError("Overpass QL must include an out statement.")
    if "{{bbox}}" in ql:
        raise PlanValidationError("Do not use the dynamic {{bbox}} placeholder.")
    has_bbox = all(str(value)[:4] in overpass_ql for value in MAINZ_BBOX.values())
    has_around = "around:" in ql
    if not has_bbox and not has_around:
        raise PlanValidationError("Overpass QL must use the Mainz bbox or an around radius.")
    if any(word in ql for word in ["germany", "deutschland", "europe"]):
        raise PlanValidationError("The query appears too broad for this Mainz exercise.")


def validate_tool_plan(plan: dict[str, Any]) -> None:
    if not isinstance(plan, dict):
        raise PlanValidationError("Tool plan must be a JSON object.")
    tool = plan.get("tool")
    if tool not in ALLOWED_TOOLS:
        raise PlanValidationError(f"Tool {tool!r} is not allowed.")
    if not isinstance(plan.get("question"), str) or not plan["question"].strip():
        raise PlanValidationError("Plan must include the original question.")

    if tool in {"overpass_search", "osm_buffer_search", "osm_nearest"}:
        validate_osm_tags(plan.get("osm_tags", {}))
        validate_overpass_ql(plan.get("overpass_ql", ""))
    if tool in {"osm_buffer_search", "osm_nearest"}:
        validate_center(plan.get("center", {}))
    if tool == "osm_buffer_search":
        radius_m = int(plan.get("radius_m", 0))
        if radius_m <= 0 or radius_m > 5000:
            raise PlanValidationError("radius_m must be between 1 and 5000.")
    if tool == "osm_nearest":
        limit = int(plan.get("limit", 5))
        if limit <= 0 or limit > 20:
            raise PlanValidationError("limit must be between 1 and 20.")
    if tool == "ors_route":
        if plan.get("start_place_id") not in KNOWN_PLACES or plan.get("end_place_id") not in KNOWN_PLACES:
            raise PlanValidationError("Route endpoints must use known Mainz place IDs.")
        if plan.get("profile") not in {"foot-walking", "cycling-regular", "driving-car"}:
            raise PlanValidationError("Unsupported ORS route profile.")


## 6. Overpass and GeoDataFrame Helpers

In [ ]:
def run_overpass_query(overpass_ql: str, pause_seconds: float = 1.0) -> dict[str, Any]:
    time.sleep(pause_seconds)
    response = requests.post(OVERPASS_URL, data={"data": overpass_ql}, timeout=60)
    if not response.ok:
        print(response.text[:1000])
        response.raise_for_status()
    return response.json()


def osm_elements_to_gdf(overpass_json: dict[str, Any]) -> gpd.GeoDataFrame:
    rows = []
    for element in overpass_json.get("elements", []):
        tags = element.get("tags", {}) or {}
        lat = element.get("lat")
        lon = element.get("lon")
        if lat is None or lon is None:
            center = element.get("center") or {}
            lat = center.get("lat")
            lon = center.get("lon")
        if lat is None or lon is None:
            continue
        row = {
            "osm_type": element.get("type"),
            "osm_id": element.get("id"),
            "name": tags.get("name"),
            "geometry": Point(float(lon), float(lat)),
        }
        row.update(tags)
        rows.append(row)
    if not rows:
        return gpd.GeoDataFrame(columns=["osm_type", "osm_id", "name", "geometry"], geometry="geometry", crs=WGS84)
    return gpd.GeoDataFrame(rows, geometry="geometry", crs=WGS84)


def make_overpass_query(osm_tags: dict[str, Any], bbox: dict[str, float] | None = None) -> str:
    bbox = bbox or MAINZ_BBOX
    bbox_text = f"{bbox['south']},{bbox['west']},{bbox['north']},{bbox['east']}"
    filters = []
    for key, value in osm_tags.items():
        filters.append(f'["{key}"]' if value is True else f'["{key}"="{value}"]')
    tag_filter = "".join(filters)
    return f"""
[out:json][timeout:25];
(
  node{tag_filter}({bbox_text});
  way{tag_filter}({bbox_text});
  relation{tag_filter}({bbox_text});
);
out center tags;
""".strip()


def show_table(gdf: gpd.GeoDataFrame, n: int = 10) -> None:
    if gdf.empty:
        display(Markdown("No features returned."))
        return
    preferred = ["name", "amenity", "tourism", "shop", "highway", "distance_m"]
    columns = [col for col in preferred if col in gdf.columns]
    display(gdf[columns].head(n) if columns else gdf.drop(columns="geometry").head(n))


## 7. Spatial Analysis and Maps

In [ ]:
def center_to_point(center: dict[str, Any]) -> Point:
    return Point(float(center["lon"]), float(center["lat"]))


def add_distance_to_center(gdf: gpd.GeoDataFrame, center: dict[str, Any]) -> gpd.GeoDataFrame:
    out = gdf.copy()
    if out.empty:
        out["distance_m"] = pd.Series(dtype="float64")
        return out
    center_gdf = gpd.GeoDataFrame(geometry=[center_to_point(center)], crs=WGS84).to_crs(METRIC_CRS)
    metric = out.to_crs(METRIC_CRS)
    out["distance_m"] = metric.distance(center_gdf.geometry.iloc[0]).round(1).values
    return out


def filter_within_radius(gdf: gpd.GeoDataFrame, center: dict[str, Any], radius_m: int) -> gpd.GeoDataFrame:
    out = add_distance_to_center(gdf, center)
    if out.empty:
        return out
    return out[out["distance_m"] <= radius_m].sort_values("distance_m").reset_index(drop=True)


def nearest_features(gdf: gpd.GeoDataFrame, center: dict[str, Any], limit: int = 5) -> gpd.GeoDataFrame:
    out = add_distance_to_center(gdf, center)
    if out.empty:
        return out
    return out.sort_values("distance_m").head(limit).reset_index(drop=True)


def summarize_gdf(gdf: gpd.GeoDataFrame) -> str:
    if gdf.empty:
        return "No OSM features were returned. Try a broader tag or larger radius."
    named = int(gdf["name"].notna().sum()) if "name" in gdf.columns else 0
    parts = [f"Returned {len(gdf)} OSM features.", f"Named features: {named}."]
    if "distance_m" in gdf.columns and gdf["distance_m"].notna().any():
        parts.append(f"Nearest: {gdf['distance_m'].min():.0f} m.")
        parts.append(f"Farthest listed: {gdf['distance_m'].max():.0f} m.")
    return " ".join(parts)


def make_base_map(title: str | None = None) -> folium.Map:
    m = folium.Map(location=MAINZ_CENTER, zoom_start=13, tiles="OpenStreetMap", control_scale=True)
    if title:
        html = f'<div style="position: fixed; top: 10px; left: 50px; z-index: 9999; background: white; padding: 8px 10px; border: 1px solid #999; font-size: 14px;"><strong>{title}</strong></div>'
        m.get_root().html.add_child(folium.Element(html))
    return m


def add_gdf_to_map(m: folium.Map, gdf: gpd.GeoDataFrame) -> folium.Map:
    if gdf.empty:
        return m
    cluster = MarkerCluster(name="OSM features").add_to(m)
    for _, row in gdf.iterrows():
        lon, lat = row.geometry.x, row.geometry.y
        parts = []
        if pd.notna(row.get("name")):
            parts.append(str(row.get("name")))
        for key in ["amenity", "tourism", "shop", "leisure", "highway"]:
            if key in row and pd.notna(row.get(key)):
                parts.append(f"{key}={row.get(key)}")
        if "distance_m" in row and pd.notna(row.get("distance_m")):
            parts.append(f"distance={row.get('distance_m'):.0f} m")
        popup = "<br>".join(parts) if parts else f"OSM {row.get('osm_type')} {row.get('osm_id')}"
        folium.CircleMarker((lat, lon), radius=5, color="blue", fill=True, fill_opacity=0.75, popup=popup).add_to(cluster)
    return m


def add_center_to_map(m: folium.Map, center: dict[str, Any], radius_m: int | None = None) -> folium.Map:
    lat, lon = float(center["lat"]), float(center["lon"])
    folium.Marker((lat, lon), popup=center.get("label", "Center"), icon=folium.Icon(color="red")).add_to(m)
    if radius_m:
        folium.Circle((lat, lon), radius=radius_m, color="red", fill=False).add_to(m)
    return m


def display_result_map(gdf: gpd.GeoDataFrame, plan: dict[str, Any]) -> folium.Map:
    m = make_base_map(plan.get("map_title", "Mainz OSM result"))
    if "center" in plan:
        add_center_to_map(m, plan["center"], plan.get("radius_m"))
    add_gdf_to_map(m, gdf)
    folium.LayerControl().add_to(m)
    display(m)
    return m


## 8. Optional OpenRouteService Tool

Create an API key at https://openrouteservice.org/dev/#/signup if you want to test routing. Without a key, the route example is skipped.

In [ ]:
ORS_API_KEY = getpass.getpass("Optional: paste your OpenRouteService API key, or press Enter to skip routing: ").strip()
print("ORS key loaded." if ORS_API_KEY else "No ORS key loaded. Routing will be skipped.")


In [ ]:
ORS_BASE_URL = "https://api.openrouteservice.org/v2/directions"


def run_ors_route(start_place_id: str, end_place_id: str, profile: str = "foot-walking") -> dict[str, Any]:
    if not ORS_API_KEY:
        raise RuntimeError("No ORS API key is loaded.")
    start = KNOWN_PLACES[start_place_id]
    end = KNOWN_PLACES[end_place_id]
    url = f"{ORS_BASE_URL}/{profile}/geojson"
    payload = {"coordinates": [[start["lon"], start["lat"]], [end["lon"], end["lat"]]]}
    response = requests.post(url, headers={"Authorization": ORS_API_KEY, "Content-Type": "application/json"}, json=payload, timeout=60)
    if not response.ok:
        print(response.text[:1000])
        response.raise_for_status()
    return response.json()


def route_geojson_to_summary(route_json: dict[str, Any]) -> dict[str, Any]:
    feature = route_json["features"][0]
    summary = feature["properties"].get("summary", {})
    return {
        "distance_m": float(summary.get("distance", 0)),
        "duration_s": float(summary.get("duration", 0)),
        "geometry": shape(feature["geometry"]),
    }


def display_route_map(route_json: dict[str, Any], plan: dict[str, Any]) -> folium.Map:
    route = route_geojson_to_summary(route_json)
    m = make_base_map(plan.get("map_title", "Mainz route"))
    start = KNOWN_PLACES[plan["start_place_id"]]
    end = KNOWN_PLACES[plan["end_place_id"]]
    folium.Marker((start["lat"], start["lon"]), popup=start["label"], icon=folium.Icon(color="green")).add_to(m)
    folium.Marker((end["lat"], end["lon"]), popup=end["label"], icon=folium.Icon(color="red")).add_to(m)
    coords = [(lat, lon) for lon, lat in route["geometry"].coords]
    folium.PolyLine(coords, color="blue", weight=5, opacity=0.8).add_to(m)
    display(Markdown(f"Distance: {route['distance_m'] / 1000:.2f} km. Duration: {route['duration_s'] / 60:.1f} minutes."))
    display(m)
    return m


## 9. Execute a Validated Plan

In [ ]:
def execute_tool_plan(plan: dict[str, Any], display_outputs: bool = True) -> dict[str, Any]:
    validate_tool_plan(plan)
    tool = plan["tool"]

    if tool in {"overpass_search", "osm_buffer_search", "osm_nearest"}:
        overpass_json = run_overpass_query(plan["overpass_ql"])
        gdf = osm_elements_to_gdf(overpass_json)
        if tool == "osm_buffer_search":
            gdf = filter_within_radius(gdf, plan["center"], int(plan["radius_m"]))
        elif tool == "osm_nearest":
            gdf = nearest_features(gdf, plan["center"], int(plan.get("limit", 5)))
        result = {"plan": plan, "gdf": gdf, "summary": summarize_gdf(gdf)}
        if display_outputs:
            display(Markdown(f"### {plan.get('map_title', 'Result')}"))
            display(Markdown(result["summary"]))
            show_table(gdf)
            result["map"] = display_result_map(gdf, plan)
        return result

    if tool == "ors_route":
        route_json = run_ors_route(plan["start_place_id"], plan["end_place_id"], plan.get("profile", "foot-walking"))
        result = {"plan": plan, "route_json": route_json, "route_summary": route_geojson_to_summary(route_json)}
        if display_outputs:
            result["map"] = display_route_map(route_json, plan)
        return result

    raise PlanValidationError(f"Unhandled tool: {tool}")


## 10. Manual Test

Inspect the JSON plan before executing it.

In [ ]:
question = "Find cafes within 800 meters of Mainz Hauptbahnhof."
plan = generate_tool_plan(question)
preview_json(plan)


In [ ]:
result = execute_tool_plan(plan)


## 11. Conversational Widget

In [ ]:
def answer_geospatial_question(question: str) -> dict[str, Any]:
    plan = generate_tool_plan(question)
    display(Markdown("### Generated tool plan"))
    preview_json(plan)
    return execute_tool_plan(plan)


if widgets is None:
    display(Markdown("ipywidgets is not available. Use `answer_geospatial_question(question)` manually."))
else:
    question_box = widgets.Textarea(
        value="Which bicycle parking locations are near Johannes Gutenberg University Mainz?",
        placeholder="Ask a Mainz OSM or route question...",
        description="Question:",
        layout=widgets.Layout(width="100%", height="90px"),
    )
    run_button = widgets.Button(description="Run", button_style="primary")
    output = widgets.Output()

    def on_run_clicked(_button):
        output.clear_output()
        with output:
            try:
                answer_geospatial_question(question_box.value)
            except Exception as exc:
                display(Markdown(f"**Error:** `{type(exc).__name__}: {exc}`"))

    run_button.on_click(on_run_clicked)
    display(widgets.VBox([question_box, run_button, output]))


## 12. Validated Examples

The tests check whether generated plans are valid and whether live API results are plausible. They do not require exact OSM feature counts.

In [ ]:
def assert_nonempty_gdf(result: dict[str, Any]) -> None:
    assert "gdf" in result, "Expected a GeoDataFrame result."
    assert not result["gdf"].empty, "Expected at least one OSM feature."


def assert_features_inside_mainz(gdf: gpd.GeoDataFrame, margin: float = 0.05) -> None:
    assert not gdf.empty, "GeoDataFrame is empty."
    minx, miny, maxx, maxy = gdf.total_bounds
    assert MAINZ_BBOX["west"] - margin <= minx <= MAINZ_BBOX["east"] + margin
    assert MAINZ_BBOX["west"] - margin <= maxx <= MAINZ_BBOX["east"] + margin
    assert MAINZ_BBOX["south"] - margin <= miny <= MAINZ_BBOX["north"] + margin
    assert MAINZ_BBOX["south"] - margin <= maxy <= MAINZ_BBOX["north"] + margin


def assert_distance_column(result: dict[str, Any]) -> None:
    gdf = result["gdf"]
    assert "distance_m" in gdf.columns, "Expected a distance_m column."
    assert gdf["distance_m"].notna().all(), "Expected all distances to be defined."


def generate_and_test(question: str, checks: list[Any]) -> dict[str, Any]:
    display(Markdown(f"### Test question: {question}"))
    plan = generate_tool_plan(question)
    preview_json(plan)
    result = execute_tool_plan(plan, display_outputs=True)
    for check in checks:
        check(result)
    display(Markdown("Validation checks passed."))
    return result


### Example 1: Cafes Near Mainz Hauptbahnhof

In [ ]:
example_1 = generate_and_test(
    "Find cafes within 800 meters of Mainz Hauptbahnhof.",
    checks=[
        assert_nonempty_gdf,
        lambda result: assert_features_inside_mainz(result["gdf"]),
        assert_distance_column,
    ],
)


### Example 2: Bicycle Parking Near JGU

This may be generated as `amenity=bicycle_parking`. If the model uses another key, update the plan or the allowed key list as part of the exercise.

In [ ]:
def assert_at_most_five_features(result: dict[str, Any]) -> None:
    assert len(result["gdf"]) <= 5, "Expected at most five nearest features."


example_2 = generate_and_test(
    "Show the five nearest bicycle parking locations to Johannes Gutenberg University Mainz.",
    checks=[
        assert_nonempty_gdf,
        assert_distance_column,
        assert_at_most_five_features,
    ],
)


### Example 3: Walking Route from Mainz Hbf to JGU

This example is skipped if no ORS key is loaded.

In [ ]:
def test_route_hbf_to_jgu() -> dict[str, Any] | None:
    if not ORS_API_KEY:
        display(Markdown("Skipping ORS route test because no ORS API key is loaded."))
        return None
    plan = generate_tool_plan("Show a walking route from Mainz Hauptbahnhof to Johannes Gutenberg University Mainz.")
    preview_json(plan)
    result = execute_tool_plan(plan, display_outputs=True)
    summary = result["route_summary"]
    assert 1500 <= summary["distance_m"] <= 7000, "Route distance should be plausible for Hbf to JGU."
    assert summary["duration_s"] > 0, "Route duration should be positive."
    display(Markdown("Validation checks passed."))
    return result


example_3 = test_route_hbf_to_jgu()


### Example 4: Guardrail Test for an Over-Broad Query

The assistant should not execute a Germany-wide OSM query.

In [ ]:
def test_guardrail_too_broad() -> None:
    question = "Find all restaurants in Germany."
    try:
        plan = generate_tool_plan(question)
        preview_json(plan)
        validate_tool_plan(plan)
        ql = plan.get("overpass_ql", "").lower()
        assert "germany" not in ql and "deutschland" not in ql, "Plan still contains a country-wide query."
        display(Markdown("Plan was accepted because it was narrowed to the Mainz study area."))
    except PlanValidationError as exc:
        display(Markdown(f"Validation rejected the broad query as expected: `{exc}`"))


test_guardrail_too_broad()


## 13. Student Tasks

1. Add a new prompt that searches for museums in Mainz using `tourism=museum`.
2. Add a prompt that searches for supermarkets using `shop=supermarket`.
3. Modify the system prompt so that the model always includes `map_title` and `analysis`.
4. Add a new validation check that rejects plans without a Mainz-specific place or bounding box.
5. Add one more spatial-analysis helper, for example `count_features_within_radius`.
6. Write two tests for your new helper.
7. Compare one successful and one failed LLM plan. Explain what failed and how validation helped.

## 14. Reflection Questions

1. Why should the LLM return a JSON plan instead of executable Python code?
2. Which parts of the workflow are deterministic, and which parts are probabilistic?
3. What can go wrong when the LLM generates Overpass QL?
4. Why do we use a projected CRS for distance calculations?
5. How would you make this widget safer before using it with non-technical users?